# Práctica 2 · Clasificación de la calidad del vino con scikit-learn

**3201 Modelos Inteligentes de Aprendizaje / 529 Redes Neuronales Modulares** · Facultad de Ingeniería,
Arquitectura y Diseño · Universidad Autónoma de Baja California · Ciclo 2026-2

**Profesores:** Dr. Everardo Inzunza González · Dr. Enrique Efrén García Guerrero · Dr. Oscar Adrián Aguirre Castro

Versión 2 · **una sesión de laboratorio de 2 horas** (Bloques A y B) **+ 2 horas de trabajo en casa** (Bloque C)

---

Este cuaderno ejecuta, paso a paso, lo que describe el manual de la práctica. Cada celda de código lleva a la
derecha de cada instrucción un comentario breve que dice **qué hace y para qué sirve**. Las salidas marcadas
con **`← Anote`** son las que usted copia en las tablas del manual (Sección 7).

**Cómo usarlo**

- Ejecute las celdas **en orden**. Si cambia algo, vuelva a ejecutar desde la celda que cambió.
- Funciona en **Google Colab** (nada que instalar) y en **Jupyter local** con Anaconda. No hace falta GPU:
  scikit-learn trabaja en CPU y todo el cuaderno termina en pocos minutos.
- En Colab: **Archivo → Guardar una copia en Drive** antes de empezar, o perderá su trabajo al cerrar.
- Los dos archivos de datos (`winequality-red.csv` y `winequality-white.csv`) están en Classroom. La Fase 0
  los busca en la carpeta actual; si no los encuentra, los descarga del repositorio UCI.

| Bloque | Qué hace | Dónde |
|---|---|---|
| **A** · Vino tinto | Análisis exploratorio → preprocesamiento → SVM, MLP y Random Forest → métricas → validación cruzada | Laboratorio, Fases 1–4 |
| **B** · Vino blanco | El mismo flujo, encapsulado en funciones: cambia el archivo y compara | Laboratorio, Fase 5 |
| **C** · Ampliaciones | Duplicados, conjunto combinado, multiclase, búsqueda de hiperparámetros, curvas de aprendizaje | Casa, 2 h |

---
## Fase 0 · Entorno, versiones y datos  *(5 min)*

Reproducibilidad es poder repetir el número. Para eso hay que dejar constancia de **con qué versiones** se
corrió y **con qué semilla**. Esta celda hace las dos cosas y debe terminar **sin errores** antes de seguir.

In [9]:
import sys, os, time, json, hashlib, zipfile, urllib.request, warnings   # utilidades del sistema y de descarga
from pathlib import Path                                                 # rutas independientes del sistema operativo

import numpy as np                                                       # cálculo numérico
import pandas as pd                                                      # tablas de datos (DataFrame)
import matplotlib.pyplot as plt                                          # gráficas
import seaborn as sns                                                    # gráficas estadísticas sobre matplotlib
import sklearn                                                           # la biblioteca de aprendizaje automático de la práctica

warnings.filterwarnings("ignore", category=FutureWarning)                # silencia sólo avisos de versiones futuras; los de convergencia SÍ se muestran
T_INICIO = time.perf_counter()                                           # cronómetro del cuaderno completo (se lee al final de cada bloque)
sns.set_theme(style="whitegrid", palette="deep")                         # estilo uniforme para todas las figuras

SEMILLA = 42                                                             # una sola semilla para todo el cuaderno
np.random.seed(SEMILLA)                                                  # fija el azar de numpy

CARPETA_FIGURAS = Path("figuras_practica2")                              # aquí se guardan las figuras para el reporte
CARPETA_FIGURAS.mkdir(exist_ok=True)                                     # se crea si no existe; no borra nada

print("Python       :", sys.version.split()[0])                          # ← Anote (Tabla A)
print("scikit-learn :", sklearn.__version__)                             # ← Anote (Tabla A)
print("pandas       :", pd.__version__)                                  # ← Anote (Tabla A)
print("numpy        :", np.__version__)
print("matplotlib   :", plt.matplotlib.__version__)
print("seaborn      :", sns.__version__)
try:
    import google.colab                                                  # este módulo sólo existe dentro de Colab
    EN_COLAB = True
except ImportError:
    EN_COLAB = False
print("En Colab     :", EN_COLAB)                                        # ← Anote (Tabla A): entorno de cómputo

Python       : 3.12.15
scikit-learn : 1.9.1
pandas       : 3.0.6
numpy        : 2.5.3
matplotlib   : 3.11.2
seaborn      : 0.13.2
En Colab     : False


**Los datos.** Son los dos archivos del conjunto *Wine Quality* del repositorio UCI (Cortez et al., 2009): 1 599
vinos tintos y 4 898 blancos de la región portuguesa del *Vinho Verde*, con 11 mediciones fisicoquímicas por vino y
una calificación sensorial de **calidad** entre 3 y 9 dada por catadores. Licencia CC BY 4.0.

La celda siguiente busca los archivos junto al cuaderno (o en `datasets_pract2/`); si no están, descarga el `.zip`
oficial de UCI. Imprime el **resumen MD5** de cada archivo para que compruebe que trabaja con la misma copia que
sus compañeros.

In [12]:
URL_UCI = "https://archive.ics.uci.edu/static/public/186/wine+quality.zip"   # archivo oficial del repositorio UCI
ARCHIVOS = ["winequality-red.csv", "winequality-white.csv"]                # los dos conjuntos de la práctica

def localizar_datos():
    """Devuelve la carpeta que contiene los dos CSV; los descarga de UCI si hacen falta."""
    for carpeta in [Path("."), Path("datasets_pract2"), Path("/content"), Path("datos_vino")]:
        if all((carpeta / a).exists() for a in ARCHIVOS):                # ¿están los dos archivos aquí?
            return carpeta
    destino = Path("datos_vino"); destino.mkdir(exist_ok=True)           # carpeta de trabajo para la descarga
    print("Descargando de UCI...", URL_UCI)
    urllib.request.urlretrieve(URL_UCI, destino / "wine_quality.zip")    # descarga (~90 KB)
    with zipfile.ZipFile(destino / "wine_quality.zip") as z:
        z.extractall(destino)                                             # extrae los CSV y el archivo .names
    return destino

DATOS = localizar_datos()                                                # carpeta con los dos CSV
print("Ruta de los datos:", DATOS.resolve())                             # ← Anote (Tabla A): ruta resuelta
for a in ARCHIVOS:
    md5 = hashlib.md5((DATOS / a).read_bytes()).hexdigest()              # huella del archivo: detecta copias alteradas
    print(f"{a:24s} {os.path.getsize(DATOS / a):>8,d} bytes   MD5 {md5}")   # ← Anote (Tabla A): MD5 de cada archivo

Ruta de los datos: C:\Users\Diana\git\Course_RedesNeuronales_2026\PR2_Wineclassification
winequality-red.csv        85,741 bytes   MD5 6d2d97f94e77df4b300dbdba8b627e76
winequality-white.csv     269,127 bytes   MD5 8a1bb2694b7680655433950a8207f9f6


> **Punto de verificación 0.** La celda anterior imprimió versiones, la ruta de los datos y dos huellas MD5 sin
> ningún error. Si algo falló (módulo faltante, descarga bloqueada), avise al profesor **antes** de continuar.

---
---
# BLOQUE A · Vino tinto

## Fase 1 · Análisis exploratorio de los datos (EDA)  *(20 min)*

Antes de entrenar nada hay que **conocer los datos**. El análisis exploratorio (Tukey, 1977) responde preguntas
concretas: ¿cuántas filas y columnas hay?, ¿de qué tipo?, ¿faltan valores?, ¿hay filas repetidas?, ¿cómo se reparte
la variable que quiero predecir?, ¿qué variables se relacionan con ella? Cada respuesta condiciona una decisión
posterior: qué escalar, cómo partir, qué métrica creer.

**Trampa frecuente:** estos CSV usan **punto y coma** como separador. Con `pd.read_csv` sin `sep=";"` obtendrá
una sola columna con todo pegado.

In [17]:
tinto = pd.read_csv(DATOS / "winequality-red.csv", sep=",")             # carga el CSV; el separador es ';'
print("Filas y columnas:", tinto.shape)                                  # ← Anote (Tabla A): (filas, columnas)
tinto.head()                                                             # primeras 5 filas: nombres y escala de cada variable

Filas y columnas: (1599, 12)


,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5


In [18]:
tinto.info()                                                             # tipo de dato y no nulos por columna: ¿todo numérico?

<class 'pandas.DataFrame'>
RangeIndex: 1599 entries, 0 to 1598
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   fixed acidity         1599 non-null   float64
 1   volatile acidity      1599 non-null   float64
 2   citric acid           1599 non-null   float64
 3   residual sugar        1599 non-null   float64
 4   chlorides             1599 non-null   float64
 5   free sulfur dioxide   1599 non-null   float64
 6   total sulfur dioxide  1599 non-null   float64
 7   density               1599 non-null   float64
 8   pH                    1599 non-null   float64
 9   sulphates             1599 non-null   float64
 10  alcohol               1599 non-null   float64
 11  quality               1599 non-null   int64  
dtypes: float64(11), int64(1)
memory usage: 150.0 KB


In [19]:
print("Valores faltantes :", int(tinto.isna().sum().sum()))              # ← Anote (Tabla A)
print("Filas duplicadas  :", int(tinto.duplicated().sum()))              # ← Anote (Tabla A): filas idénticas a otra anterior
tinto.describe().T.round(3)                                              # media, desviación, mínimo, cuartiles y máximo por variable

Valores faltantes : 0
Filas duplicadas  : 240


,count,mean,std,min,25%,50%,75%,max
fixed acidity,1599.0,8.320,1.741,4.600,7.100,7.900,9.200,15.900
volatile acidity,1599.0,0.528,0.179,0.120,0.390,0.520,0.640,1.580
citric acid,1599.0,0.271,0.195,0.000,0.090,0.260,0.420,1.000
residual sugar,1599.0,2.539,1.410,0.900,1.900,2.200,2.600,15.500
chlorides,1599.0,0.087,0.047,0.012,0.070,0.079,0.090,0.611
free sulfur dioxide,1599.0,15.875,10.460,1.000,7.000,14.000,21.000,72.000
total sulfur dioxide,1599.0,46.468,32.895,6.000,22.000,38.000,62.000,289.000
density,1599.0,0.997,0.002,0.990,0.996,0.997,0.998,1.004
pH,1599.0,3.311,0.154,2.740,3.210,3.310,3.400,4.010
sulphates,1599.0,0.658,0.170,0.330,0.550,0.620,0.730,2.000


**Lea la tabla `describe()` con estas preguntas:** ¿las variables tienen escalas muy distintas (compare `total sulfur
dioxide` con `density`)? Eso es lo que obliga a **escalar** antes de la SVM y del MLP. ¿Algún máximo está muy lejos del
tercer cuartil? Ahí hay valores atípicos que conviene mirar en los histogramas.

In [16]:
conteo = tinto["quality"].value_counts().sort_index()                    # cuántos vinos hay por cada nota de calidad
porcentaje = (100 * conteo / len(tinto)).round(1)                        # lo mismo, en porcentaje
print(pd.DataFrame({"vinos": conteo, "%": porcentaje}))                  # ← Anote (Tabla A): distribución de 'quality'

fig, ax = plt.subplots(figsize=(6, 3.4))
sns.countplot(x="quality", data=tinto, color="#17795C", ax=ax)          # barras: una por nota de calidad
ax.set_title("Vino tinto: distribución de la calidad (n = %d)" % len(tinto))
ax.set_xlabel("calidad sensorial (3–8)"); ax.set_ylabel("número de vinos")
plt.tight_layout(); fig.savefig(CARPETA_FIGURAS / "F1_tinto_distribucion_calidad.png", dpi=150)   # Figura F1 del reporte
plt.show()

KeyError: 'quality'

> **Fíjese en el desbalance.** Casi todos los vinos tienen calidad 5 o 6; los extremos son raros. Esto va a
> decidir qué métrica es creíble más adelante: un clasificador que diga siempre «calidad normal» acertará la mayoría
> de las veces sin haber aprendido nada.

In [ ]:
variables = [col for col in tinto.columns if col != "quality"]           # las 11 mediciones fisicoquímicas
fig, ejes = plt.subplots(3, 4, figsize=(13, 8))
for ax, col in zip(ejes.flat, variables):
    sns.histplot(tinto[col], bins=30, color="#17795C", ax=ax)           # histograma: forma, sesgo y valores atípicos
    ax.set_title(col, fontsize=10); ax.set_xlabel(""); ax.set_ylabel("")
ejes.flat[-1].axis("off")                                                # el duodécimo panel queda vacío
fig.suptitle("Vino tinto: distribución de cada variable", y=1.0)
plt.tight_layout(); fig.savefig(CARPETA_FIGURAS / "F2_tinto_histogramas.png", dpi=150)   # Figura F2
plt.show()

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(11, 3.8))
sns.boxplot(x="quality", y="alcohol", data=tinto, color="#D9A21B", ax=ejes[0])            # ¿sube el alcohol con la calidad?
sns.boxplot(x="quality", y="volatile acidity", data=tinto, color="#17795C", ax=ejes[1])   # ¿baja la acidez volátil?
ejes[0].set_title("alcohol por nivel de calidad"); ejes[1].set_title("acidez volátil por nivel de calidad")
plt.tight_layout(); fig.savefig(CARPETA_FIGURAS / "F3_tinto_boxplots.png", dpi=150)       # Figura F3
plt.show()

In [ ]:
corr = tinto.corr(numeric_only=True)                                     # matriz de correlación de Pearson entre las 12 columnas
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdYlGn", center=0, square=True,
            cbar_kws={"shrink": 0.8}, annot_kws={"size": 8}, ax=ax)     # mapa de calor: rojo negativo, verde positivo
ax.set_title("Vino tinto: correlación entre variables")
plt.tight_layout(); fig.savefig(CARPETA_FIGURAS / "F4_tinto_correlacion.png", dpi=150)    # Figura F4
plt.show()

corr_calidad = corr["quality"].drop("quality").sort_values(key=np.abs, ascending=False)   # ordena por |r|
print("Correlación con la calidad (de mayor a menor en valor absoluto):")
print(corr_calidad.round(3).to_string())                                 # ← Anote (Tabla B): las tres primeras

> **Punto de verificación 1.** Llene la **Tabla A** (filas, columnas, faltantes, duplicados, distribución de
> `quality`) y la **Tabla B** (las tres variables más correlacionadas con la calidad y su signo). Guarde las
> Figuras F1 a F4. Anote una hipótesis: ¿qué variable cree que el modelo usará más?

---
## Fase 2 · Preprocesamiento  *(15 min)*

Tres decisiones, cada una con su razón:

1. **Convertir la calidad en dos clases.** La calidad va de 3 a 8 y las notas extremas tienen muy pocos casos; con seis
   clases tan desbalanceadas las clases raras casi no se pueden aprender ni evaluar (lo comprobará en C3). Binarizar
   **simplifica la tarea y cambia la pregunta**: ya no «qué nota tiene» sino «¿es de calidad alta?». Se sigue la convención
   del video de apoyo: **calidad alta si `quality ≥ 7`** (clase 1), **normal en caso contrario** (clase 0). El video lo hace
   con `pd.cut` y `LabelEncoder`; aquí se hace en una línea y, en la celda siguiente, se comprueba que ambas rutas dan lo
   mismo. El umbral es una variable, pero cambiarlo define **otra tarea** con otra prevalencia: sus métricas no son
   comparables con las de esta.
2. **Partir en entrenamiento y prueba de forma estratificada** (80 / 20), para que ambas partes conserven
   aproximadamente el mismo porcentaje de vinos de calidad alta (con conteos enteros no puede ser exacto). La partición de
   prueba **no se toca** hasta evaluar.
3. **Escalar las variables** con `StandardScaler` (media 0, desviación 1), ajustado **sólo con el entrenamiento**.
   Si se ajusta con todos los datos, la prueba «contamina» al modelo con información que no debería tener: eso se llama
   **fuga de información** y produce métricas infladas.

In [ ]:
UMBRAL = 7                                                               # calidad alta si quality >= 7 (variable: pruébelo en el Bloque C)
X = tinto.drop(columns="quality")                                        # las 11 variables de entrada
y = (tinto["quality"] >= UMBRAL).astype(int)                             # objetivo binario: 1 = calidad alta, 0 = normal

print("Clase 1 (calidad alta) :", int(y.sum()), f"vinos  ({100 * y.mean():.1f} %)")        # ← Anote (Tabla A)
print("Clase 0 (normal)       :", int((y == 0).sum()), f"vinos  ({100 * (1 - y.mean()):.1f} %)")
print(f"Un modelo que SIEMPRE diga 'normal' acertaría el {100 * (1 - y.mean()):.1f} % de las veces.")   # ← Anote: exactitud trivial

In [ ]:
# La misma binarización tal como la hace el video de apoyo: pd.cut con los cortes (2, 6.5, 8) y LabelEncoder.
from sklearn.preprocessing import LabelEncoder                          # convierte etiquetas de texto en enteros

calidad_texto = pd.cut(tinto["quality"], bins=(2, 6.5, 8), labels=["bad", "good"])   # (2, 6.5] -> bad ; (6.5, 8] -> good
y_video = LabelEncoder().fit_transform(calidad_texto.astype(str))       # 'bad' -> 0, 'good' -> 1 (orden alfabético)
print("¿pd.cut del video == (quality >= 7)?", bool(np.array_equal(y_video, y.values)))   # debe imprimir True
print("Vinos fuera de los cortes del video (NaN):", int(calidad_texto.isna().sum()))     # 0 en tinto; en blanco NO sería 0

> **Por qué aquí se usa `quality >= 7` y no los cortes del video.** Los intervalos son `(2, 6.5]` → `bad` y `(6.5, 8]` →
> `good`; toda calidad mayor que 8 queda **fuera** y recibe `NaN` (la fila no se borra, pero pierde su etiqueta). En el
> tinto no hay ninguna (máximo 8); en el blanco hay cinco vinos con calidad 9 que, al codificar con `LabelEncoder`
> tras `astype(str)`, se convertirían en una **tercera clase espuria** `"nan"`. La comparación directa no tiene ese hueco.

In [ ]:
from sklearn.model_selection import train_test_split                    # partición aleatoria reproducible

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=SEMILLA)             # 80/20, estratificada por la clase, semilla fija

print("Entrenamiento:", X_train.shape, "| calidad alta:", f"{100 * y_train.mean():.1f} %")   # ← Anote (Tabla A)
print("Prueba       :", X_test.shape,  "| calidad alta:", f"{100 * y_test.mean():.1f} %")    # ← Anote (Tabla A)

# ¿Cuántas filas de prueba tienen las 11 mediciones IDÉNTICAS a alguna fila de entrenamiento? (duplicados que cruzan la partición)
# Aquí se comparan sólo las 11 variables de entrada; `tinto.duplicated()` (Fase 1) comparaba además la calidad.
repetidas = X_test.merge(X_train.drop_duplicates(), how="inner").shape[0]   # coincidencias exactas de las 11 variables
print(f"Filas de prueba con las 11 variables repetidas en entrenamiento: {repetidas} de {len(X_test)}")   # ← Anote (Tabla A)

> **Ojo con ese último número.** Si una fila de prueba es idéntica a una de entrenamiento, el modelo ya la «vio»
> y la métrica de prueba queda un poco inflada. En el Bloque C medirá cuánto cambia el resultado si elimina los
> duplicados **antes** de partir. Aquí se conservan, como hace el conjunto original, para que sus cifras sean
> comparables con las de la literatura y con las del video de apoyo.

In [ ]:
from sklearn.preprocessing import StandardScaler                        # z = (x - media) / desviación

escalador = StandardScaler().fit(X_train)                                # aprende media y desviación SÓLO del entrenamiento
X_train_s = escalador.transform(X_train)                                 # escala el entrenamiento
X_test_s  = escalador.transform(X_test)                                  # escala la prueba con los MISMOS parámetros

comparacion = pd.DataFrame({
    "media train original": X_train.mean().round(3),
    "desv. train original": X_train.std(ddof=0).round(3),               # ddof=0: la misma convención (poblacional) que usa StandardScaler
    "media train escalada": X_train_s.mean(axis=0).round(3),
    "desv. train escalada": X_train_s.std(axis=0).round(3),
    "media PRUEBA escalada": X_test_s.mean(axis=0).round(3),              # no es 0: se escaló con la media del entrenamiento
})
comparacion                                                              # ← Anote (Tabla B): dos variables, antes y después

> **Punto de verificación 2.** Tabla A completa (porcentaje de clase alta en el total, en entrenamiento y en prueba;
> filas de prueba repetidas en entrenamiento) y Tabla B con el efecto del escalado en dos variables. Pregunta para
> responder al profesor: ¿por qué la media escalada de la prueba **no** es exactamente 0?

---
## Fase 3 · Tres clasificadores y sus métricas  *(35 min)*

Se entrenan tres familias distintas sobre los **mismos** datos escalados y se evalúan sobre la **misma** partición
de prueba. Antes, las métricas que va a leer. Las siete primeras se calculan con las **etiquetas predichas** y se derivan
de la **matriz de confusión** (VP = verdaderos positivos, FP = falsos positivos, FN = falsos negativos, VN = verdaderos
negativos; «positivo» = calidad alta). El **AUC-ROC** es distinto: usa los **puntajes** del modelo (probabilidad o
`decision_function`) y recorre todos los umbrales posibles; una matriz de confusión, que corresponde a un solo umbral,
no lo determina.

| Métrica | Fórmula | Qué responde |
|---|---|---|
| Exactitud (*accuracy*) | (VP + VN) / total | ¿Qué fracción de vinos clasificó bien? **Engaña si hay desbalance** |
| Precisión (*precision*) | VP / (VP + FP) | De los que llamó «alta», ¿cuántos lo eran? |
| Sensibilidad (*recall*) | VP / (VP + FN) | De los que eran «alta», ¿cuántos encontró? |
| Especificidad | VN / (VN + FP) | De los «normales», ¿cuántos reconoció como tales? |
| F1 | 2·Prec·Sens / (Prec + Sens) | Media armónica: castiga que una de las dos sea baja |
| Exactitud balanceada | (Sens + Espec) / 2 | Exactitud corregida por el desbalance |
| MCC | (VP·VN − FP·FN) / √[(VP+FP)(VP+FN)(VN+FP)(VN+FN)] | Correlación entre predicción y verdad, de −1 a 1; 0 = sin relación. Con una predicción constante el denominador es 0 y scikit-learn devuelve 0 por convención (Chicco y Jurman, 2020) |
| AUC-ROC | área bajo la curva sensibilidad vs. tasa de falsos positivos al variar el umbral | Capacidad de ordenar positivos antes que negativos; 0.5 = azar, 1.0 = orden perfecto. **Usa puntajes, no la matriz** |
| *support* | número de casos reales de cada clase | Aparece en `classification_report`, junto con los promedios *macro* (media simple de las clases) y *ponderado* (por *support*) |

Cuando un denominador es cero (por ejemplo la precisión de un modelo que nunca predice «alta»), scikit-learn devuelve 0
porque se le indica `zero_division=0`; no es que la precisión «valga» 0, es que no está definida.

La función siguiente entrena, predice, calcula todo esto, imprime el `classification_report` y dibuja la matriz de
confusión. Se usa igual para los tres modelos, así que **la comparación es justa**.

In [ ]:
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, precision_score, recall_score,
                             f1_score, matthews_corrcoef, roc_auc_score, confusion_matrix,
                             classification_report, ConfusionMatrixDisplay, RocCurveDisplay)

RESULTADOS = {}                                                          # aquí se acumulan las métricas de cada modelo
MODELOS = {}                                                             # y los modelos ya entrenados (para las curvas ROC)

def evaluar_clasificador(modelo, nombre, X_tr, y_tr, X_te, y_te, guardar=None):
    """Entrena `modelo`, evalúa sobre la prueba y devuelve un diccionario con las métricas."""
    t0 = time.perf_counter()
    modelo.fit(X_tr, y_tr)                                               # entrenamiento: el modelo aprende de X_tr, y_tr
    segundos = time.perf_counter() - t0                                  # tiempo de entrenamiento
    y_pred = modelo.predict(X_te)                                        # clases predichas sobre la prueba
    if hasattr(modelo, "predict_proba"):
        puntaje = modelo.predict_proba(X_te)[:, 1]                       # probabilidad de la clase 1 (para AUC-ROC)
    elif hasattr(modelo, "decision_function"):
        puntaje = modelo.decision_function(X_te)                         # distancia al hiperplano (SVM)
    else:
        puntaje = y_pred                                                 # último recurso: la propia predicción
    vn, fp, fn, vp = confusion_matrix(y_te, y_pred).ravel()              # las cuatro celdas de la matriz de confusión
    m = {
        "exactitud":            accuracy_score(y_te, y_pred),
        "exactitud balanceada": balanced_accuracy_score(y_te, y_pred),
        "precisión":            precision_score(y_te, y_pred, zero_division=0),
        "sensibilidad":         recall_score(y_te, y_pred),
        "especificidad":        float(vn / (vn + fp)) if (vn + fp) else 0.0,
        "F1":                   f1_score(y_te, y_pred),
        "MCC":                  matthews_corrcoef(y_te, y_pred),
        "AUC-ROC":              roc_auc_score(y_te, puntaje),
        "tiempo (s)":           segundos,
    }
    m = {k: float(v) for k, v in m.items()}                              # números de Python, sin envolturas de numpy
    m["VP·FP·FN·VN"] = f"{vp}·{fp}·{fn}·{vn}"                            # las cuatro celdas, para el registro
    print(f"===== {nombre} =====   entrenado en {segundos:.2f} s")
    print(f"VP={vp}  FP={fp}  FN={fn}  VN={vn}")                          # ← Anote (Tabla C): las cuatro celdas
    print(classification_report(y_te, y_pred, target_names=["normal (0)", "alta (1)"], digits=3, zero_division=0))   # precision, recall, f1, support
    print({k: (round(v, 3) if isinstance(v, float) else v) for k, v in m.items()})   # ← Anote (Tabla C)
    fig, ax = plt.subplots(figsize=(4.2, 3.6))
    ConfusionMatrixDisplay.from_predictions(y_te, y_pred, display_labels=["normal", "alta"],
                                            cmap="Greens", colorbar=False, ax=ax)   # matriz de confusión: filas = verdad, columnas = predicción
    ax.set_title(f"Matriz de confusión · {nombre}")
    plt.tight_layout()
    if guardar:
        fig.savefig(CARPETA_FIGURAS / guardar, dpi=150)                  # figura para el reporte
    plt.show()
    RESULTADOS[nombre] = m; MODELOS[nombre] = modelo
    return m

### 3.0 Línea base: el clasificador que no aprende nada

Antes de los modelos «de verdad», un **clasificador trivial** que siempre predice la clase mayoritaria. Su exactitud
es la **referencia** contra la que se lee la de cualquier otro modelo: superar ese número no garantiza utilidad, y no
superarlo tampoco la descarta si lo que importa es detectar la clase minoritaria (ahí mandan sensibilidad, F1 y MCC).

In [ ]:
from sklearn.dummy import DummyClassifier                                # clasificador trivial de referencia

evaluar_clasificador(DummyClassifier(strategy="most_frequent"),          # siempre predice la clase más frecuente
                     "Línea base (mayoritaria)", X_train_s, y_train, X_test_s, y_test);
print(f"Proporción de clase normal EN LA PRUEBA: {(y_test == 0).mean():.4f}  ({int((y_test == 0).sum())} de {len(y_test)})")   # ← Anote: compárela con la exactitud de la línea base

> La exactitud de la línea base es exactamente la proporción de vinos «normales» **en la prueba** (277 de 320 = 86.6 %
> en la corrida del profesor); no es idéntica al 86.4 % de todo el conjunto porque la estratificación trabaja con conteos
> enteros. Y mire su sensibilidad, su F1 y su MCC: **cero**. Ésa es la razón de no fiarse de la
> exactitud sola.

### 3.1 Máquina de vectores de soporte (SVM)

Busca el **hiperplano** que separa las dos clases con el **margen** más ancho posible; el núcleo RBF le permite
que esa frontera sea curva en el espacio original (Cortes y Vapnik, 1995). Dos hiperparámetros mandan: `C`
(cuánto castiga cada error de entrenamiento) y `gamma` (qué tan local es la influencia de cada punto). **Necesita
datos escalados**: la distancia entre puntos es su materia prima.

In [ ]:
from sklearn.svm import SVC                                              # Support Vector Classifier

svm = SVC(kernel="rbf", C=1.0, gamma="scale", random_state=SEMILLA)     # núcleo RBF; C y gamma en sus valores por omisión
evaluar_clasificador(svm, "SVM (RBF)", X_train_s, y_train, X_test_s, y_test,
                     guardar="F5a_tinto_confusion_svm.png");            # Figura F5a

### 3.2 Perceptrón multicapa (MLP)

Una red neuronal densa: 11 entradas → dos capas ocultas (64 y 32 neuronas, activación ReLU) → una salida. Aprende
por **retropropagación del error** (Rumelhart, Hinton y Williams, 1986) con el optimizador Adam. Es el mismo tipo de
red que entrenó en la Práctica 1 con PyTorch, ahora en dos líneas de scikit-learn. También exige datos escalados, y
su `loss_curve_` muestra cómo bajó el error durante el entrenamiento.

El video de apoyo usa tres capas de 11 neuronas y `max_iter=500`. Aquí se eligió (64, 32) con `max_iter=1000`
porque la red del video **no alcanza a converger** en 500 iteraciones con estos datos; en el Bloque C la reproducirá
y verá la advertencia con sus propios ojos.

In [ ]:
from sklearn.neural_network import MLPClassifier                         # red neuronal densa (perceptrón multicapa)

mlp = MLPClassifier(hidden_layer_sizes=(64, 32), activation="relu", solver="adam",
                    max_iter=1000, random_state=SEMILLA)                # 2 capas ocultas; hasta 1000 épocas; semilla fija
evaluar_clasificador(mlp, "MLP (64, 32)", X_train_s, y_train, X_test_s, y_test,
                     guardar="F5b_tinto_confusion_mlp.png");            # Figura F5b
print(f"Épocas ejecutadas: {mlp.n_iter_} de {mlp.max_iter} · "
      f"{'se detuvo por el criterio de tolerancia (convergió)' if mlp.n_iter_ < mlp.max_iter else 'AGOTÓ max_iter: NO convergió'}")   # ← Anote (Tabla C)

fig, ax = plt.subplots(figsize=(6, 3.2))
ax.plot(mlp.loss_curve_, color="#17795C")                                # pérdida por época: debe bajar y estabilizarse
ax.set_xlabel("época"); ax.set_ylabel("pérdida (log-loss)")
ax.set_title(f"MLP: curva de entrenamiento ({mlp.n_iter_} épocas)")    # la misma cifra impresa arriba
plt.tight_layout(); fig.savefig(CARPETA_FIGURAS / "F6_tinto_mlp_perdida.png", dpi=150)   # Figura F6
plt.show()

### 3.3 Bosque aleatorio (Random Forest)

Un **conjunto de muchos árboles de decisión**, cada uno entrenado con una muestra distinta de los datos y con un
subconjunto aleatorio de variables en cada división; la predicción final es el voto de todos (Breiman, 2001). No
necesita escalado (los árboles comparan umbrales, no distancias), pero se le pasan los datos escalados para que la
comparación con los otros dos sea idéntica. Regala una **importancia por variable**: cuánto ayudó cada una a separar
las clases.

In [ ]:
from sklearn.ensemble import RandomForestClassifier                      # bosque aleatorio

rf = RandomForestClassifier(n_estimators=200, random_state=SEMILLA, n_jobs=-1)   # 200 árboles; usa todos los núcleos
evaluar_clasificador(rf, "Random Forest (200)", X_train_s, y_train, X_test_s, y_test,
                     guardar="F5c_tinto_confusion_rf.png");             # Figura F5c

importancia = pd.Series(rf.feature_importances_, index=X.columns).sort_values()   # importancia de cada variable
fig, ax = plt.subplots(figsize=(6, 4))
importancia.plot.barh(color="#D9A21B", ax=ax)                            # barras horizontales, de menor a mayor
ax.set_title("Random Forest: importancia de las variables"); ax.set_xlabel("importancia (suma = 1)")
plt.tight_layout(); fig.savefig(CARPETA_FIGURAS / "F7_tinto_rf_importancia.png", dpi=150)   # Figura F7
plt.show()
print("Tres variables más importantes:", list(importancia.index[-3:][::-1]))   # ← Anote (Tabla B): compare con su hipótesis

In [ ]:
tabla_C_tinto = pd.DataFrame(RESULTADOS).T                               # una fila por modelo, una columna por métrica
_num = [c for c in tabla_C_tinto.columns if c != "VP·FP·FN·VN"]
tabla_C_tinto[_num] = tabla_C_tinto[_num].astype(float).round(3)
tabla_C_tinto                                                            # ← Anote (Tabla C): tabla comparativa del vino tinto

In [ ]:
fig, ax = plt.subplots(figsize=(5.6, 4.6))
for nombre in ["SVM (RBF)", "MLP (64, 32)", "Random Forest (200)"]:
    RocCurveDisplay.from_estimator(MODELOS[nombre], X_test_s, y_test, name=nombre, ax=ax)   # curva ROC de cada modelo
ax.plot([0, 1], [0, 1], "k--", lw=1, label="azar (AUC = 0.5)")          # la diagonal: un clasificador al azar
ax.set_title("Vino tinto: curvas ROC sobre la prueba"); ax.legend(fontsize=8)
plt.tight_layout(); fig.savefig(CARPETA_FIGURAS / "F8_tinto_roc.png", dpi=150)   # Figura F8
plt.show()

> **Punto de verificación 3.** Tabla C llena para los cuatro modelos (línea base incluida), Figuras F5 a F8
> guardadas. Prepare dos respuestas: (a) ¿qué modelo tiene la mejor **sensibilidad** y a qué **precisión** la
> consigue?; (b) ¿en qué orden quedan los modelos por exactitud y en qué orden por F1? Si el orden cambia, explique
> por qué.

---
## Fase 4 · Validación cruzada *k-fold*  *(20 min)*

Una sola partición de prueba da **un** número, y ese número depende de qué vinos cayeron en la prueba. La
**validación cruzada** reparte el entrenamiento en *k* pliegues, entrena *k* veces dejando fuera un pliegue distinto
cada vez y reporta la **media y la desviación** de las *k* métricas. La desviación es tan importante como la media:
dice cuánto cambia el resultado con los datos.

Dos detalles de método que marcan la diferencia:

- Se hace **sobre el entrenamiento**. La prueba ya se usó una vez en la Fase 3, con fines didácticos; no se vuelve a
  usar para decidir nada.
- El escalador va **dentro de un `Pipeline`**: así se reajusta en cada pliegue con sus propios datos y no hay fuga de
  escalado. Llamar `cross_val_score(clf, X_train_s, y_train, cv=5)` con los datos ya escalados funciona, pero filtra
  información del pliegue de validación al escalador; la diferencia aquí es pequeña, la costumbre correcta no.
- El `Pipeline` **no** evita otra dependencia: las filas duplicadas pueden caer en pliegues distintos (o en
  entrenamiento y prueba). Se conservan, como en el video, para que sus cifras sean comparables; el Bloque C1 mide el
  efecto.

Se prueban **k = 5** y **k = 10** con pliegues estratificados y barajados, y dos métricas a la vez, exactitud y F1, con
`cross_validate` (una sola ronda de *k* ajustes por modelo). Es una versión ampliada de la instrucción del curso,
`cross_val_score(clf, x_train, y_train, cv=5)`: misma idea, con el escalador dentro, dos métricas y pliegues barajados
con semilla, por lo que la partición en pliegues no es idéntica a la de la llamada literal.

In [ ]:
from sklearn.model_selection import cross_validate, StratifiedKFold     # validación cruzada estratificada, varias métricas a la vez
from sklearn.pipeline import make_pipeline                              # encadena escalador + modelo
from sklearn.base import clone                                          # copia un modelo sin entrenar

CV = {}                                                                  # resultados de validación cruzada
PLIEGUES = {}                                                            # puntajes por pliegue (para la gráfica)

def validacion_cruzada(modelo, nombre, X_tr, y_tr, ks=(5, 10)):
    """cross_validate con k = 5 y 10: escalador dentro del pipeline; exactitud y F1 de los MISMOS k ajustes."""
    fila = {}
    for k in ks:
        kf = StratifiedKFold(n_splits=k, shuffle=True, random_state=SEMILLA)   # k pliegues con la misma proporción de clases
        tubo = make_pipeline(StandardScaler(), clone(modelo))            # el escalador se ajusta dentro de cada pliegue
        t0 = time.perf_counter()
        res = cross_validate(tubo, X_tr, y_tr, cv=kf, scoring=("accuracy", "f1"))   # k ajustes; dos métricas por pliegue
        acc, f1 = res["test_accuracy"], res["test_f1"]
        fila[f"exactitud k={k}"] = f"{acc.mean():.3f} ± {acc.std():.3f}"
        fila[f"F1 k={k}"]        = f"{f1.mean():.3f} ± {f1.std():.3f}"
        fila[f"tiempo k={k} (s)"] = round(time.perf_counter() - t0, 1)
        PLIEGUES[(nombre, k)] = acc                                      # se guardan los k valores para graficarlos
        print(f"{nombre:22s} k={k:2d}  exactitud {acc.mean():.3f} ± {acc.std():.3f}   F1 {f1.mean():.3f} ± {f1.std():.3f}   ({fila[f'tiempo k={k} (s)']} s)")   # ← Anote (Tabla D)
    CV[nombre] = fila
    return fila

t_celda = time.perf_counter()                                            # cronómetro de ESTA celda completa
for nombre, modelo in [("SVM (RBF)", svm), ("MLP (64, 32)", mlp), ("Random Forest (200)", rf)]:
    validacion_cruzada(modelo, nombre, X_train, y_train)                 # sobre el entrenamiento SIN escalar: el pipeline escala
print(f"Duración de esta celda: {time.perf_counter() - t_celda:.1f} s")   # ← Anote (Tabla D): tiempo real de la celda en su equipo

In [ ]:
tabla_D_tinto = pd.DataFrame(CV).T                                       # una fila por modelo
tabla_D_tinto                                                            # ← Anote (Tabla D): media ± desviación para k=5 y k=10

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.8))
datos_caja = [PLIEGUES[(n, k)] for n in ["SVM (RBF)", "MLP (64, 32)", "Random Forest (200)"] for k in (5, 10)]
etiquetas  = [f"{n.split(' ')[0]}\ncv={k}" for n in ["SVM (RBF)", "MLP (64, 32)", "Random Forest (200)"] for k in (5, 10)]
ax.boxplot(datos_caja, tick_labels=etiquetas)                           # una caja por modelo y por k: dispersión entre pliegues
ax.axhline(1 - y_train.mean(), color="#9E2B25", ls="--", lw=1, label="clase mayoritaria en el entrenamiento")   # línea base calculada sobre los mismos datos que los pliegues
ax.set_ylabel("exactitud por pliegue"); ax.set_title("Vino tinto: validación cruzada, dispersión entre pliegues")
ax.legend(fontsize=8)
plt.tight_layout(); fig.savefig(CARPETA_FIGURAS / "F9_tinto_cv_pliegues.png", dpi=150)   # Figura F9
plt.show()

> **Punto de verificación 4.** Tabla D llena y Figura F9 guardada. Tres preguntas: ¿la exactitud de la validación
> cruzada coincide con la de la prueba (Tabla C) dentro de una desviación?; ¿qué cambió entre cv = 5 y cv = 10, la
> media o la desviación?; ¿qué modelo es más **estable** entre pliegues?

---
---
# BLOQUE B · Vino blanco

## Fase 5 · Repetir el flujo con otro conjunto  *(25 min)*

El flujo del Bloque A, **resumido**, cabe en dos funciones: `eda_resumen` (tamaño, tipos, faltantes, duplicados,
`describe`, distribución de la calidad y correlaciones) y `experimento_completo` (binarizar, partir, escalar, línea base +
tres clasificadores con `classification_report`, y validación cruzada de los tres clasificadores). No repite los
histogramas, los diagramas de caja ni la importancia de variables. Con ellas repite el experimento sobre el vino blanco
**cambiando una sola variable**: el archivo. Lea las salidas con las mismas preguntas del Bloque A.

In [ ]:
def eda_resumen(df, nombre, prefijo):
    """EDA compacto: tamaño, faltantes, duplicados, distribución de la calidad y correlaciones."""
    print(f"===== {nombre}: análisis exploratorio =====")
    print("Filas y columnas :", df.shape, "| columnas numéricas:", int(df.select_dtypes("number").shape[1]), "de", df.shape[1])   # ← Anote (Tabla E)
    print("Valores faltantes:", int(df.isna().sum().sum()), "| filas duplicadas:", int(df.duplicated().sum()))   # ← Anote (Tabla E)
    display(df.describe().T.round(3))                                    # estadísticos por variable
    conteo = df["quality"].value_counts().sort_index()
    print("Distribución de la calidad:", conteo.to_dict())               # ← Anote (Tabla E)
    corr_q = df.corr(numeric_only=True)["quality"].drop("quality").sort_values(key=np.abs, ascending=False)
    print("Tres variables más correlacionadas con la calidad:", corr_q.head(3).round(3).to_dict())   # ← Anote (Tabla E)
    fig, ejes = plt.subplots(1, 2, figsize=(13, 4.6), gridspec_kw={"width_ratios": [1, 1.6]})
    sns.countplot(x="quality", data=df, color="#17795C", ax=ejes[0]); ejes[0].set_title(f"{nombre}: calidad (n = {len(df)})")
    sns.heatmap(df.corr(numeric_only=True), annot=True, fmt=".2f", cmap="RdYlGn", center=0,
                annot_kws={"size": 6.5}, cbar=False, ax=ejes[1]); ejes[1].set_title(f"{nombre}: correlación")
    plt.tight_layout(); fig.savefig(CARPETA_FIGURAS / f"{prefijo}_eda.png", dpi=150); plt.show()
    return corr_q

def experimento_completo(df, nombre, prefijo, umbral, quitar_duplicados=False):
    """Preprocesa, entrena línea base + SVM + MLP + RF, evalúa en prueba y hace validación cruzada k = 5 y 10."""
    if quitar_duplicados:
        df = df.drop_duplicates()                                        # opcional (Bloque C): elimina filas repetidas (las 12 columnas) ANTES de partir
    X = df.drop(columns="quality"); y = (df["quality"] >= umbral).astype(int)
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.20, stratify=y, random_state=SEMILLA)
    esc = StandardScaler().fit(X_tr); X_tr_s, X_te_s = esc.transform(X_tr), esc.transform(X_te)
    repetidas = X_te.merge(X_tr.drop_duplicates(), how="inner").shape[0]   # filas de prueba con las 11 variables repetidas en entrenamiento
    print(f"===== {nombre}: umbral >= {umbral}, n = {len(df)}, calidad alta = {100 * y.mean():.1f} %, "
          f"entrenamiento {len(X_tr)}, prueba {len(X_te)}, filas de prueba repetidas en entrenamiento {repetidas} =====")   # ← Anote (Tabla E)
    modelos = {
        "Línea base (mayoritaria)": DummyClassifier(strategy="most_frequent"),
        "SVM (RBF)":                SVC(kernel="rbf", C=1.0, gamma="scale", random_state=SEMILLA),
        "MLP (64, 32)":             MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=1000, random_state=SEMILLA),
        "Random Forest (200)":      RandomForestClassifier(n_estimators=200, random_state=SEMILLA, n_jobs=-1),
    }
    metricas, cv_filas = {}, {}
    fig, ejes = plt.subplots(1, 4, figsize=(15, 3.5))
    for ax, (n, m) in zip(ejes, modelos.items()):
        t0 = time.perf_counter(); m.fit(X_tr_s, y_tr); seg = time.perf_counter() - t0
        pred = m.predict(X_te_s)
        punt = m.predict_proba(X_te_s)[:, 1] if hasattr(m, "predict_proba") else (
               m.decision_function(X_te_s) if hasattr(m, "decision_function") else pred)
        vn, fp, fn, vp = confusion_matrix(y_te, pred).ravel()
        metricas[n] = {"exactitud": accuracy_score(y_te, pred), "exactitud balanceada": balanced_accuracy_score(y_te, pred),
                       "precisión": precision_score(y_te, pred, zero_division=0), "sensibilidad": recall_score(y_te, pred),
                       "especificidad": float(vn / (vn + fp)) if (vn + fp) else 0.0, "F1": f1_score(y_te, pred),
                       "MCC": matthews_corrcoef(y_te, pred), "AUC-ROC": roc_auc_score(y_te, punt), "tiempo (s)": seg,
                       "VP·FP·FN·VN": f"{vp}·{fp}·{fn}·{vn}"}
        print(f"--- {n}: VP={vp} FP={fp} FN={fn} VN={vn}" + (f" · épocas {m.n_iter_}/{m.max_iter}" if isinstance(m, MLPClassifier) else ""))
        print(classification_report(y_te, pred, target_names=["normal (0)", "alta (1)"], digits=3, zero_division=0))   # precision, recall, f1, support
        ConfusionMatrixDisplay.from_predictions(y_te, pred, display_labels=["normal", "alta"], cmap="Greens",
                                                colorbar=False, ax=ax); ax.set_title(n, fontsize=9)
        if n != "Línea base (mayoritaria)":                              # la línea base no se valida en cruzado
            fila = {}
            for k in (5, 10):
                kf = StratifiedKFold(n_splits=k, shuffle=True, random_state=SEMILLA)
                tubo = make_pipeline(StandardScaler(), clone(m))
                t0 = time.perf_counter()
                res = cross_validate(tubo, X_tr, y_tr, cv=kf, scoring=("accuracy", "f1"))   # k ajustes, dos métricas
                acc, f1v = res["test_accuracy"], res["test_f1"]
                fila[f"exactitud k={k}"] = f"{acc.mean():.3f} ± {acc.std():.3f}"
                fila[f"F1 k={k}"] = f"{f1v.mean():.3f} ± {f1v.std():.3f}"
                fila[f"tiempo k={k} (s)"] = round(time.perf_counter() - t0, 1)
            cv_filas[n] = fila
    fig.suptitle(f"{nombre}: matrices de confusión sobre la prueba", y=1.02)
    plt.tight_layout(); fig.savefig(CARPETA_FIGURAS / f"{prefijo}a_confusiones.png", dpi=150); plt.show()   # subfigura a
    fig, ax = plt.subplots(figsize=(5.2, 4.2))
    for n in ["SVM (RBF)", "MLP (64, 32)", "Random Forest (200)"]:
        RocCurveDisplay.from_estimator(modelos[n], X_te_s, y_te, name=n, ax=ax)
    ax.plot([0, 1], [0, 1], "k--", lw=1); ax.set_title(f"{nombre}: curvas ROC"); ax.legend(fontsize=8)
    plt.tight_layout(); fig.savefig(CARPETA_FIGURAS / f"{prefijo}b_roc.png", dpi=150); plt.show()   # subfigura b
    tabla = pd.DataFrame(metricas).T
    tabla[[c for c in tabla.columns if c != "VP·FP·FN·VN"]] = tabla[[c for c in tabla.columns if c != "VP·FP·FN·VN"]].astype(float).round(3)
    return tabla, pd.DataFrame(cv_filas).T

print("Funciones listas: eda_resumen() y experimento_completo().")

In [ ]:
blanco = pd.read_csv(DATOS / "winequality-white.csv", sep=";")           # el segundo conjunto: 4 898 vinos blancos
corr_blanco = eda_resumen(blanco, "Vino blanco", "F10_blanco")           # Figura F10 (calidad + correlación)

In [ ]:
t_celda = time.perf_counter()                                            # cronómetro de ESTA celda completa
tabla_F_blanco, tabla_F_blanco_cv = experimento_completo(blanco, "Vino blanco", "F11_blanco", umbral=UMBRAL)   # Figuras F11a y F11b
print(f"Duración de esta celda: {time.perf_counter() - t_celda:.1f} s")   # ← Anote (Tabla F): tiempo real de la celda en su equipo
print("Métricas sobre la prueba:"); display(tabla_F_blanco)              # ← Anote (Tabla F)
print("Validación cruzada (tres clasificadores):"); display(tabla_F_blanco_cv)   # ← Anote (Tabla F)

In [ ]:
comparativa = pd.concat({"Tinto": tabla_C_tinto, "Blanco": tabla_F_blanco}, axis=0)   # las dos tablas, una debajo de la otra
print(f"Tiempo transcurrido desde la Fase 0: {(time.perf_counter() - T_INICIO) / 60:.1f} min")   # ← Anote (Tabla A): duración real de los Bloques A y B en su equipo
comparativa[["exactitud", "exactitud balanceada", "precisión", "sensibilidad", "F1", "MCC", "AUC-ROC", "tiempo (s)"]]   # ← va a la sección Resultados del reporte técnico

> **Punto de verificación 5 (cierre de la sesión).** Tablas E y F llenas, comparativa tinto/blanco completa,
> Figuras F10 y F11 guardadas. Para la discusión: ¿el mejor modelo es el mismo en los dos vinos?; ¿en cuál de los dos
> conjuntos es más difícil detectar la calidad alta, y qué dato del EDA lo anticipaba?

### Antes de cerrar: guarde sus resultados

La celda siguiente escribe un archivo `resultados_practica2.json` con las versiones, las huellas MD5, el análisis
exploratorio, las métricas de prueba (con las cuatro celdas de cada matriz), la validación cruzada por pliegue y los
tiempos, y comprime la carpeta de figuras. En Colab además descarga ambos archivos; el almacenamiento de Colab **se borra** al cerrar la sesión.

In [ ]:
import shutil
REGISTRO = {"versiones": {"python": sys.version.split()[0], "scikit-learn": sklearn.__version__, "pandas": pd.__version__,
                          "numpy": np.__version__, "en_colab": EN_COLAB},
            "semilla": SEMILLA, "umbral": UMBRAL, "ruta_datos": str(DATOS.resolve()),
            "md5": {a: hashlib.md5((DATOS / a).read_bytes()).hexdigest() for a in ARCHIVOS},
            "eda_tinto": {"forma": list(tinto.shape), "faltantes": int(tinto.isna().sum().sum()), "duplicados": int(tinto.duplicated().sum()),
                          "calidad": {int(k): int(v) for k, v in tinto["quality"].value_counts().sort_index().items()},
                          "clase_alta": int(y.sum()), "prueba_repetidas_en_train": int(repetidas),
                          "correlacion_con_calidad": {k: float(v) for k, v in corr_calidad.items()},
                          "escalado": comparacion.to_dict(), "importancia_rf": {k: float(v) for k, v in importancia.items()}},
            "eda_blanco": {"forma": list(blanco.shape), "faltantes": int(blanco.isna().sum().sum()), "duplicados": int(blanco.duplicated().sum()),
                           "calidad": {int(k): int(v) for k, v in blanco["quality"].value_counts().sort_index().items()},
                           "correlacion_con_calidad": {k: float(v) for k, v in corr_blanco.items()}},
            "tinto_prueba": pd.DataFrame(RESULTADOS).T.to_dict(), "tinto_cv": tabla_D_tinto.to_dict(),
            "tinto_cv_pliegues": {f"{n} k={k}": [float(v) for v in a] for (n, k), a in PLIEGUES.items()},
            "mlp_epocas": int(mlp.n_iter_), "mlp_max_iter": int(mlp.max_iter),
            "blanco_prueba": tabla_F_blanco.to_dict(), "blanco_cv": tabla_F_blanco_cv.to_dict(),
            "minutos_bloques_A_B": round((time.perf_counter() - T_INICIO) / 60, 2)}

def exportar(etiqueta):
    """Escribe el JSON y comprime la carpeta de figuras; en Colab además los descarga."""
    Path("resultados_practica2.json").write_text(json.dumps(REGISTRO, indent=1, ensure_ascii=False, default=str), encoding="utf-8")
    shutil.make_archive("figuras_practica2", "zip", CARPETA_FIGURAS)     # comprime todas las figuras presentes
    print(f"[{etiqueta}] Guardados: resultados_practica2.json y figuras_practica2.zip "
          f"({len(list(CARPETA_FIGURAS.glob('*.png')))} figuras: {sorted(p.name for p in CARPETA_FIGURAS.glob('*.png'))})")
    if EN_COLAB:
        from google.colab import files                                   # sólo existe en Colab
        files.download("resultados_practica2.json"); files.download("figuras_practica2.zip")
    else:
        print("Entorno local: los archivos están en", Path(".").resolve())

exportar("cierre de la sesión")

---
---
# BLOQUE C · Trabajo en casa  *(2 horas)*

Cinco ampliaciones cortas, cada una responde una pregunta que quedó abierta en el laboratorio. Registre los
resultados en las Tablas G a I del manual y úselos en la discusión de su reporte técnico.

| Ampliación | Pregunta | Tiempo |
|---|---|---|
| C1 · Duplicados | ¿Cambian las métricas si elimino las filas repetidas antes de partir? | 15 min |
| C2 · Conjunto combinado | ¿Es fácil distinguir tinto de blanco? ¿Mejora la predicción de la calidad saber el tipo? | 20 min |
| C3 · Multiclase | ¿Qué pasa si intento predecir las seis notas de calidad en vez de dos clases? | 15 min |
| C4 · Hiperparámetros | ¿Cuánto mejora la SVM si ajusto `C` y `gamma`? ¿Qué le pasa al MLP del video? | 20 min |
| C5 · Curvas de aprendizaje | ¿Le faltan datos al modelo o le sobra capacidad? | 10 min |
| Reporte técnico | Redacción con las tablas y figuras ya llenas durante la sesión | 40 min |

> **Sobre el protocolo.** En la sesión comparó los modelos sobre la partición de prueba con fines didácticos, así que
> esa prueba ya no es una evaluación independiente. En un trabajo real las decisiones (qué modelo, qué hiperparámetros)
> se toman con validación **dentro del entrenamiento** y la prueba se usa **una sola vez**, al cierre. En este bloque,
> toda comparación entre variantes se decide con la validación cruzada; las cifras de prueba que siga imprimiendo son
> exploratorias y se reportan como tales.

## C1 · El efecto de los duplicados

En la Fase 2 contó cuántas filas de prueba tenían sus 11 variables repetidas en entrenamiento. Ahora elimine las
filas duplicadas (las 12 columnas idénticas) **antes** de partir y repita el experimento completo. Compare exactitud,
F1 y MCC con la Tabla C. Ojo: al quitar filas cambian el tamaño, la partición y la prueba misma, así que la diferencia
que observe es **entre dos protocolos**, no una medida exacta de cuánto «inflaba» la fuga.

In [ ]:
print("Duplicados en tinto :", int(tinto.duplicated().sum()), "de", len(tinto))
print("Duplicados en blanco:", int(blanco.duplicated().sum()), "de", len(blanco))
tabla_G_sin_dup, tabla_G_sin_dup_cv = experimento_completo(tinto, "Vino tinto SIN duplicados", "F12_tinto_sin_dup",
                                                           umbral=UMBRAL, quitar_duplicados=True)   # ← Anote (Tabla G): compare con la Tabla C
display(tabla_G_sin_dup); display(tabla_G_sin_dup_cv)
REGISTRO["C1_sin_duplicados"] = {"prueba": tabla_G_sin_dup.to_dict(), "cv": tabla_G_sin_dup_cv.to_dict()}

> Lo que obtiene es la **diferencia entre dos protocolos** (con y sin duplicados), no la magnitud de una causa. Una
> caída es compatible con que las filas repetidas inflaban la prueba, pero también con que la prueba ahora es otra y más
> pequeña; y una diferencia pequeña no demuestra que el efecto de la fuga sea pequeño, porque varias causas pueden
> compensarse. Reporte la diferencia observada y las explicaciones compatibles. Compare también la validación cruzada,
> que promedia sobre más particiones.

## C2 · Conjunto combinado: dos preguntas distintas

Se unen los dos archivos y se añade la columna `tipo` (0 = tinto, 1 = blanco). Primero un problema **fácil**:
¿puede un Random Forest distinguir tinto de blanco a partir de la química? Después el problema **difícil** de siempre,
la calidad alta, ahora con `tipo` como variable adicional: ¿la usa el modelo?

In [ ]:
tinto_t = tinto.assign(tipo=0); blanco_t = blanco.assign(tipo=1)         # etiqueta de tipo de vino
combinado = pd.concat([tinto_t, blanco_t], ignore_index=True)            # 6 497 vinos
print("Combinado:", combinado.shape, "| blancos:", f"{100 * combinado['tipo'].mean():.1f} %")   # ← Anote (Tabla H)

# (a) Problema fácil: predecir el TIPO de vino a partir de las 11 variables químicas
X_t = combinado.drop(columns=["quality", "tipo"]); y_t = combinado["tipo"]
Xa, Xb, ya, yb = train_test_split(X_t, y_t, test_size=0.20, stratify=y_t, random_state=SEMILLA)
rf_tipo = RandomForestClassifier(n_estimators=200, random_state=SEMILLA, n_jobs=-1).fit(Xa, ya)
print(f"Tinto vs blanco · exactitud = {accuracy_score(yb, rf_tipo.predict(Xb)):.4f}  "
      f"F1 = {f1_score(yb, rf_tipo.predict(Xb)):.4f}")                  # ← Anote (Tabla H)
print("Variables que más separan tinto de blanco:",
      list(pd.Series(rf_tipo.feature_importances_, index=X_t.columns).sort_values(ascending=False).index[:3]))   # ← Anote

# (b) Problema difícil: calidad alta en el conjunto combinado, con 'tipo' como variable de entrada
tabla_H_combinado, tabla_H_combinado_cv = experimento_completo(combinado, "Combinado (con 'tipo')", "F13_combinado", umbral=UMBRAL)   # ← Anote (Tabla H)
display(tabla_H_combinado)
REGISTRO["C2_tipo_de_vino"] = {"exactitud": float(accuracy_score(yb, rf_tipo.predict(Xb))), "F1": float(f1_score(yb, rf_tipo.predict(Xb)))}
REGISTRO["C2_combinado"] = {"prueba": tabla_H_combinado.to_dict(), "cv": tabla_H_combinado_cv.to_dict()}

**¿Ayuda `tipo` a predecir la calidad?** La pregunta correcta se responde comparando **el mismo modelo, sobre la misma
partición, con y sin la columna**; la importancia por impureza sólo dice cuánto la usó el bosque, y una variable
redundante con otras (aquí la química ya separa tinto de blanco) puede pesar poco aunque «contenga» la información.

In [ ]:
X_c = combinado.drop(columns="quality"); y_c = (combinado["quality"] >= UMBRAL).astype(int)
Xa, Xb, ya, yb = train_test_split(X_c, y_c, test_size=0.20, stratify=y_c, random_state=SEMILLA)   # UNA partición para ambas variantes
rf_con = RandomForestClassifier(n_estimators=200, random_state=SEMILLA, n_jobs=-1)                 # con 'tipo'
rf_sin = RandomForestClassifier(n_estimators=200, random_state=SEMILLA, n_jobs=-1)                 # sin 'tipo'
# 1) La decisión se toma por validación cruzada dentro del entrenamiento...
kf5 = StratifiedKFold(5, shuffle=True, random_state=SEMILLA)
f1_con = cross_validate(clone(rf_con), Xa, ya, cv=kf5, scoring="f1")["test_score"]
f1_sin = cross_validate(clone(rf_sin), Xa.drop(columns="tipo"), ya, cv=kf5, scoring="f1")["test_score"]
print(f"F1 en validación cruzada (k=5) · con 'tipo' {f1_con.mean():.3f} ± {f1_con.std():.3f} · sin 'tipo' {f1_sin.mean():.3f} ± {f1_sin.std():.3f}")   # ← Anote (Tabla H)
# 2) ...y la prueba se reporta después, como cifra exploratoria (esta partición ya se usó en la sesión).
rf_con.fit(Xa, ya); rf_sin.fit(Xa.drop(columns="tipo"), ya)
reg_tipo = {"f1_cv_con": float(f1_con.mean()), "f1_cv_con_std": float(f1_con.std()), "f1_cv_sin": float(f1_sin.mean()), "f1_cv_sin_std": float(f1_sin.std())}
for etiqueta, m, Xe in [("con 'tipo'", rf_con, Xb), ("sin 'tipo'", rf_sin, Xb.drop(columns="tipo"))]:
    pr = m.predict(Xe)
    print(f"RF {etiqueta:10s} (prueba, exploratorio) · exactitud {accuracy_score(yb, pr):.3f}  F1 {f1_score(yb, pr):.3f}  MCC {matthews_corrcoef(yb, pr):.3f}")   # ← Anote (Tabla H)
    reg_tipo[etiqueta] = {"exactitud": float(accuracy_score(yb, pr)), "F1": float(f1_score(yb, pr)), "MCC": float(matthews_corrcoef(yb, pr))}
imp_c = pd.Series(rf_con.feature_importances_, index=X_c.columns).sort_values(ascending=False)   # del MISMO bosque evaluado
print("Importancia de 'tipo' en ese bosque:", round(float(imp_c["tipo"]), 4),
      "| posición:", int(list(imp_c.index).index("tipo")) + 1, "de", len(imp_c))   # ← Anote (Tabla H)
reg_tipo.update({"importancia_tipo": float(imp_c["tipo"]), "posicion_tipo": int(list(imp_c.index).index("tipo")) + 1})
REGISTRO["C2_con_vs_sin_tipo"] = reg_tipo

> Un problema con exactitud cercana a 1.0 y otro con F1 mucho menor sobre **los mismos datos**: la dificultad no está
> en el algoritmo sino en la pregunta. Y si con y sin `tipo` el F1 apenas cambia, en **este** experimento (este bosque,
> esta partición, estos pliegues) no se observa una mejora apreciable; una explicación compatible es que la química
> (dióxido de azufre, cloruros, acidez volátil) ya contiene la información del tipo. No es una prueba general.

## C3 · ¿Y si no binarizamos? Clasificación multiclase

Se entrena el Random Forest para predecir directamente la nota de calidad (3 a 8 en tinto). Mire la exactitud, la
exactitud balanceada (en multiclase es el **promedio de la sensibilidad de cada clase**) y la matriz de confusión de
6 × 6: ¿qué clases acierta y cuáles nunca?

In [ ]:
y_multi = tinto["quality"]                                               # objetivo con seis clases (3 a 8)
Xa, Xb, ya, yb = train_test_split(tinto.drop(columns="quality"), y_multi, test_size=0.20,
                                  stratify=y_multi, random_state=SEMILLA)
rf_multi = RandomForestClassifier(n_estimators=200, random_state=SEMILLA, n_jobs=-1).fit(Xa, ya)
pred = rf_multi.predict(Xb)
print(f"Multiclase · exactitud = {accuracy_score(yb, pred):.3f}   exactitud balanceada = {balanced_accuracy_score(yb, pred):.3f}")   # ← Anote (Tabla G)
REGISTRO["C3_multiclase"] = {"exactitud": float(accuracy_score(yb, pred)), "exactitud_balanceada": float(balanced_accuracy_score(yb, pred))}
print(classification_report(yb, pred, zero_division=0, digits=3))      # observe el 'support' de las clases 3, 4 y 8
fig, ax = plt.subplots(figsize=(5, 4.4))
ConfusionMatrixDisplay.from_predictions(yb, pred, cmap="Greens", colorbar=False, ax=ax)
ax.set_title("Tinto, multiclase: matriz de confusión 6 × 6"); plt.tight_layout()
fig.savefig(CARPETA_FIGURAS / "F14_tinto_multiclase.png", dpi=150); plt.show()   # Figura F14

> La exactitud multiclase parece «aceptable» y la balanceada es muy baja: el modelo acierta las clases 5 y 6 y casi
> nunca las raras. Ésa es la razón práctica de binarizar; en su reporte diga también qué se pierde al hacerlo.

## C4 · Ajuste de hiperparámetros de la SVM con `GridSearchCV`

En la Fase 3 la SVM usó `C = 1` y `gamma = "scale"` por omisión. `GridSearchCV` prueba una rejilla de valores con
validación cruzada **dentro del entrenamiento**, elige la mejor combinación por F1 y sólo entonces se evalúa en la
prueba. Compare con la SVM de la Tabla C.

In [ ]:
from sklearn.model_selection import GridSearchCV                         # búsqueda exhaustiva con validación cruzada

rejilla = {"svc__C": [0.1, 1, 10, 100], "svc__gamma": ["scale", 0.01, 0.1, 1]}   # 16 combinaciones
tubo_svm = make_pipeline(StandardScaler(), SVC(kernel="rbf", random_state=SEMILLA))
busqueda = GridSearchCV(tubo_svm, rejilla, scoring="f1",
                        cv=StratifiedKFold(5, shuffle=True, random_state=SEMILLA), n_jobs=-1)
t0 = time.perf_counter(); busqueda.fit(X_train, y_train); print(f"búsqueda en {time.perf_counter() - t0:.1f} s")
print("Mejores parámetros:", busqueda.best_params_, "| F1 medio en validación:", round(busqueda.best_score_, 3))   # ← Anote (Tabla I)
pred = busqueda.predict(X_test)                                          # el pipeline ganador, ya reentrenado con todo el entrenamiento
print(f"SVM ajustada sobre la prueba · exactitud = {accuracy_score(y_test, pred):.3f}   "
      f"F1 = {f1_score(y_test, pred):.3f}   MCC = {matthews_corrcoef(y_test, pred):.3f}")   # ← Anote (Tabla I): compare con la Tabla C
print(f"Para comparar por validación: F1 k=5 de la SVM por omisión = {CV['SVM (RBF)']['F1 k=5']} frente a {busqueda.best_score_:.3f} de la ajustada")
REGISTRO["C4_gridsearch_svm"] = {"mejores": {k: (v if isinstance(v, str) else float(v)) for k, v in busqueda.best_params_.items()},
                                 "f1_validacion": float(busqueda.best_score_), "exactitud_prueba": float(accuracy_score(y_test, pred)),
                                 "f1_prueba": float(f1_score(y_test, pred)), "mcc_prueba": float(matthews_corrcoef(y_test, pred))}

**C4b · El MLP del video.** Tres capas ocultas de 11 neuronas y `max_iter=500`, como en el tutorial. Lea el texto de la
advertencia de convergencia si aparece, cuántas iteraciones hizo y cómo quedan exactitud y F1 frente al MLP (64, 32) de la
Tabla C. Si no convergió, pruebe `max_iter=2000` y compare las dos variantes **por validación cruzada** (como en la
Fase 4), no sobre la prueba.

In [ ]:
from sklearn.exceptions import ConvergenceWarning                        # la advertencia que emite un MLP que no converge

with warnings.catch_warnings(record=True) as avisos:                     # captura las advertencias en vez de ocultarlas
    warnings.simplefilter("always", ConvergenceWarning)
    mlp_video = MLPClassifier(hidden_layer_sizes=(11, 11, 11), max_iter=500, random_state=SEMILLA).fit(X_train_s, y_train)
convergio = not any(issubclass(a.category, ConvergenceWarning) for a in avisos)
for a in avisos:
    print("ADVERTENCIA:", str(a.message)[:200])                            # el texto que scikit-learn emite cuando se agota max_iter
pred = mlp_video.predict(X_test_s)
print(f"MLP del video (11, 11, 11), max_iter=500 · iteraciones = {mlp_video.n_iter_} · convergió = {convergio}")   # ← Anote (Tabla I)
print(f"prueba (exploratorio): exactitud = {accuracy_score(y_test, pred):.3f}   F1 = {f1_score(y_test, pred):.3f}   "
      f"MCC = {matthews_corrcoef(y_test, pred):.3f}")                    # ← Anote (Tabla I): compare con el MLP (64, 32)
# La comparación que decide: validación cruzada (k=5) de las dos arquitecturas, con el escalador dentro del pipeline.
with warnings.catch_warnings():
    warnings.simplefilter("ignore", ConvergenceWarning)                 # aquí ya sabemos que la del video no converge
    f1_video = cross_validate(make_pipeline(StandardScaler(), MLPClassifier(hidden_layer_sizes=(11, 11, 11), max_iter=500, random_state=SEMILLA)),
                              X_train, y_train, cv=StratifiedKFold(5, shuffle=True, random_state=SEMILLA), scoring="f1")["test_score"]
print(f"F1 en validación cruzada (k=5) · MLP del video {f1_video.mean():.3f} ± {f1_video.std():.3f} · MLP (64, 32) {CV['MLP (64, 32)']['F1 k=5']}")   # ← Anote (Tabla I)
REGISTRO["C4b_mlp_video"] = {"iteraciones": int(mlp_video.n_iter_), "convergio": bool(convergio), "exactitud_prueba": float(accuracy_score(y_test, pred)),
                             "f1_prueba": float(f1_score(y_test, pred)), "mcc_prueba": float(matthews_corrcoef(y_test, pred)),
                             "f1_cv_video": float(f1_video.mean()), "f1_cv_video_std": float(f1_video.std())}

## C5 · Curvas de aprendizaje: ¿faltan datos o sobra capacidad?

La curva de aprendizaje entrena con fracciones crecientes del entrenamiento y mide una métrica en entrenamiento y
en validación cruzada. Se usa **F1**, no la exactitud, porque con este desbalance la exactitud sube sola con la clase
mayoritaria. Ojo: un F1 positivo tampoco demuestra por sí solo que el modelo discrimine; un modelo que dijera siempre
«alta» tendría F1 = 2p/(1+p) ≈ 0.24 con p = 13.6 % de positivos. La referencia útil es ese valor, no el 0 de la línea
base mayoritaria. Lectura, como **hipótesis** a contrastar: si la curva de entrenamiento queda muy por encima de la de validación, hay indicios de **sobreajuste**;
si ambas quedan bajas y juntas, de **subajuste**; si la de validación todavía sube en el último tramo, es plausible
que más datos ayuden.

In [ ]:
from sklearn.model_selection import LearningCurveDisplay                 # curva de aprendizaje lista para graficar

fig, ejes = plt.subplots(1, 2, figsize=(11, 3.8))
REGISTRO["C5_curvas_aprendizaje"] = {}
for ax, (nombre, modelo) in zip(ejes, [("Random Forest (200)", rf), ("MLP (64, 32)", mlp)]):
    disp = LearningCurveDisplay.from_estimator(make_pipeline(StandardScaler(), clone(modelo)), X_train, y_train,
                                               train_sizes=np.linspace(0.1, 1.0, 6),
                                               cv=StratifiedKFold(5, shuffle=True, random_state=SEMILLA),
                                               scoring="f1", n_jobs=-1, ax=ax)   # 6 tamaños (10 % a 100 %) × 5 pliegues
    ax.set_title(f"Curva de aprendizaje (F1) · {nombre}"); ax.set_ylim(0, 1.02)
    tr, va = disp.train_scores.mean(axis=1), disp.test_scores.mean(axis=1)   # medias por tamaño de entrenamiento
    print(f"{nombre:20s} F1 con el máximo de filas de entrenamiento por pliegue ({int(disp.train_sizes[-1])}): "
          f"entrenamiento {tr[-1]:.3f} · validación {va[-1]:.3f} · validación con {int(disp.train_sizes[-2])} filas: {va[-2]:.3f}")   # ← Anote: brecha y tendencia del último tramo
    REGISTRO["C5_curvas_aprendizaje"][nombre] = {"tamaños": [int(t) for t in disp.train_sizes], "f1_train": [float(v) for v in tr], "f1_val": [float(v) for v in va]}
plt.tight_layout(); fig.savefig(CARPETA_FIGURAS / "F15_tinto_curvas_aprendizaje.png", dpi=150); plt.show()   # Figura F15

> **Diagnóstico para el reporte:** ¿cuál de los dos modelos muestra una brecha mayor entre entrenamiento y
> validación? ¿La curva de validación sigue subiendo en el último tramo (entre los dos tamaños mayores impresos; el 100 % es el máximo de filas de entrenamiento **dentro de cada pliegue**, unas cuatro quintas partes de `X_train`)? Responda
> con las cifras impresas y la figura, como hipótesis que otro conjunto de datos podría contradecir.

### Exportación final (Bloque C incluido)

La exportación del cierre de la sesión no contenía las figuras F12 a F15 ni los resultados del Bloque C. Esta sí.

In [ ]:
REGISTRO["minutos_cuaderno_completo"] = round((time.perf_counter() - T_INICIO) / 60, 2)   # duración total en su equipo
print(f"Tiempo total del cuaderno: {REGISTRO['minutos_cuaderno_completo']} min")            # ← Anote: reporte técnico, Materiales y métodos
exportar("cierre del Bloque C")

---
## Cierre · Reporte técnico

Redacte su reporte con esta estructura (Sección 7.3 del manual): **Resumen** (150 palabras) · **Introducción** (el
problema, el conjunto y las tres familias de clasificadores, con citas) · **Materiales y métodos** (entorno, versiones,
semilla, umbral, partición, escalado, hiperparámetros y protocolo de validación) · **Resultados** (Tablas A–I y las
figuras que las sustentan) · **Interpretación y discusión** (qué modelo elegiría y por qué; qué métrica manda con
este desbalance; qué dice la validación cruzada sobre la estabilidad; qué mostró el Bloque C) · **Conclusiones**
propias · **Referencias** en formato APA · **Declaración de uso de IA** (Sección 7.2 del manual).

**Lo que se entrega** (evidencias del manual): **E1** este cuaderno ejecutado, Bloques A a C, con todas las salidas
visibles · **E2** `resultados_practica2.json` y `figuras_practica2.zip` de la exportación final · **E3** el manual con las
Tablas A a I llenas y las seis figuras de la Sección 7.1 · **E4** el reporte técnico en PDF con cuestionario, conclusiones
y declaración de uso de IA.

### Créditos de los datos y del software

- **Datos.** Cortez, P., Cerdeira, A., Almeida, F., Matos, T., y Reis, J. (2009). Modeling wine preferences by data
  mining from physicochemical properties. *Decision Support Systems, 47*(4), 547–553.
  https://doi.org/10.1016/j.dss.2009.05.016 — Conjunto *Wine Quality* del UCI Machine Learning Repository,
  licencia CC BY 4.0: https://archive.ics.uci.edu/dataset/186/wine+quality
- **Software.** Pedregosa, F., et al. (2011). Scikit-learn: Machine learning in Python. *Journal of Machine Learning
  Research, 12*, 2825–2830. https://jmlr.org/papers/v12/pedregosa11a.html — Documentación: https://scikit-learn.org/stable/

*Elaboraron: Dr. Everardo Inzunza González · Dr. Enrique Efrén García Guerrero · Dr. Oscar Adrián Aguirre Castro ·
FIAD-UABC · Ciclo 2026-2*